# 01 Distal Credit Ladder

All seven figures are constructed inline from their authored source generators. The three result panels compute fresh samples by default; archived numerical results are opt-in. Figures always display once, while external saving remains disabled by default.

## Configuration

Shared run, device, worker, archive, and saving controls.

In [ ]:
from pathlib import Path
import hashlib, json, os, sys

# Keep BLAS libraries single-threaded when independent cells are process-parallel.
for _name in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
    os.environ.setdefault(_name, "1")

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

RUN_PROFILE = os.getenv("MRL_RUN_PROFILE", "reduced")  # reduced | publication | smoke
DEVICE = os.getenv("MRL_DEVICE", "auto")               # NumPy reference algorithms remain on CPU
WORKERS = os.getenv("MRL_WORKERS", "auto")
SAVE_FIGURES = os.getenv("MRL_SAVE_FIGURES", "0") == "1"
OUTPUT_DIR = Path(os.getenv("MRL_OUTPUT_DIR", "generated_figures"))
OVERWRITE = os.getenv("MRL_OVERWRITE", "0") == "1"
RUN_EXTERNAL_DATA = os.getenv("MRL_RUN_EXTERNAL_DATA", "0") == "1"
ALLOW_DATA_DOWNLOADS = os.getenv("MRL_ALLOW_DATA_DOWNLOADS", "0") == "1"
USE_ARCHIVED_RESULTS = os.getenv("MRL_USE_ARCHIVED_RESULTS", "0") == "1"

if RUN_PROFILE not in {"reduced", "publication", "smoke"}:
    raise ValueError(f"unknown RUN_PROFILE={RUN_PROFILE!r}")

HERE = Path.cwd()
ROOT = HERE.parent if HERE.name == "experiments" else HERE
OWNED = ("fig_architecture.png", "fig_bio_silicon_map.png", "fig_crossbar_rl.png",
         "fig_eligibility_electron.png", "fig_rl_curve.png",
         "fig_tier1_window.png", "fig_tier2_saturation.png")
FIGURE_REPORT = []
SELECTED_DEVICE = "cpu"

if os.getenv("MRL_CHILD_PROCESS", "0") == "1":
    RESOLVED_WORKERS = 1  # the reproduction driver already owns the outer pool
elif WORKERS == "auto":
    RESOLVED_WORKERS = min(6, max(1, (os.cpu_count() or 2) // 2))
else:
    RESOLVED_WORKERS = max(1, int(WORKERS))

PROFILE = {
    "smoke": dict(tier1_seeds=1, tier2_seeds=1, tier2_trials=4, tier2_dt=1e-2,
                  rl_seeds=2, rl_trials=100),
    "reduced": dict(tier1_seeds=6, tier2_seeds=4, tier2_trials=240, tier2_dt=5e-3,
                    rl_seeds=6, rl_trials=600),
    "publication": dict(tier1_seeds=20, tier2_seeds=20, tier2_trials=400, tier2_dt=1e-3,
                        rl_seeds=20, rl_trials=600),
}[RUN_PROFILE]

print({"profile": RUN_PROFILE, "requested_device": DEVICE, "selected_device": SELECTED_DEVICE,
       "workers": RESOLVED_WORKERS, "saving": SAVE_FIGURES,
       "archived_results": USE_ARCHIVED_RESULTS, "interpreter": sys.executable})

## Inline display, saving, and provenance

In [ ]:
def _jsonable(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, dict):
        return {str(key): _jsonable(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [_jsonable(item) for item in value]
    return value


def _data_hash(value):
    if value is None:
        return None
    payload = json.dumps(_jsonable(value), sort_keys=True, separators=(",", ":"), allow_nan=False)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def _safe_save(fig, filename, provenance, bbox_inches=None):
    if not SAVE_FIGURES:
        return None
    if provenance in {"reference-only", "placeholder", "external-gated"}:
        raise ValueError(f"{provenance} outputs cannot be published: {filename}")
    out = OUTPUT_DIR.expanduser()
    if not out.is_absolute():
        out = ROOT / out
    out = out.resolve()
    if "manuscript" in str(out).lower() and not OVERWRITE:
        raise FileExistsError("writing into a manuscript directory requires OVERWRITE=True")
    out.mkdir(parents=True, exist_ok=True)
    path = out / filename
    if path.exists() and not OVERWRITE:
        raise FileExistsError(path)
    fig.savefig(path, dpi=300, bbox_inches=bbox_inches, facecolor="white")
    return str(path)


def finish(fig, filename, *, provenance, claim_status, data=None, seeds=None, source_mode="live", bbox_inches=None):
    """Display exactly once, optionally save, close, and record provenance."""
    if filename not in OWNED:
        raise KeyError(f"unregistered notebook figure: {filename}")
    saved = _safe_save(fig, filename, provenance, bbox_inches=bbox_inches)
    record = {
        "filename": filename,
        "tier": "notebook-owned",
        "profile": RUN_PROFILE,
        "device": SELECTED_DEVICE,
        "seeds": seeds,
        "data_hash": _data_hash(data),
        "reference_hash": None,
        "provenance_class": provenance,
        "claim_status": claim_status,
        "source_mode": source_mode,
        "saved_path": saved,
        "method_provenance": {"status": "proposed",
            "established_basis": ["three-factor reward modulation", "eligibility traces"],
            "repository_adaptation": "notebook-owned MRL ladder",
            "claim_limit": "repository implementation, not an exact cited plasticity rule"},
    }
    FIGURE_REPORT.append(record)
    display(fig)
    plt.close(fig)
    return record


def _archive(name):
    path = ROOT / "data" / "results" / name
    if not path.exists():
        return None
    return np.load(path, allow_pickle=True).item()

## Three-factor architecture

Authored schematic generator; no raster fallback is used.

In [ ]:
import os
import numpy as np
import matplotlib

import matplotlib.pyplot as plt
from matplotlib.patches import (Circle, FancyBboxPatch, FancyArrowPatch,
                                Rectangle)

INK = "#2b2b2b"; CUE = "#e0a93b"; DIST = "#9aa6b2"; REW = "#c0392b"
TRACE = "#3aa07a"; DEV = "#eaf0fb"; OUT = "#cfd8e0"

plt.rcParams.update({"font.family": "DejaVu Sans"})
fig = plt.figure(figsize=(10.0, 4.4))
gsL = fig.add_axes([0.015, 0.04, 0.52, 0.92]); gsL.axis("off")
gsR = fig.add_axes([0.60, 0.04, 0.39, 0.92]); gsR.axis("off")
gsL.set_xlim(0, 10.6); gsL.set_ylim(0, 10)
gsR.set_xlim(0, 10); gsR.set_ylim(0, 10)

def arrow(ax, p0, p1, color=INK, lw=1.8, ms=12, ls="-", z=5, rad=0.0):
    cs = f"arc3,rad={rad}" if rad else None
    ax.add_patch(FancyArrowPatch(p0, p1, arrowstyle="-|>", mutation_scale=ms,
                 color=color, lw=lw, ls=ls, zorder=z, shrinkA=0, shrinkB=0,
                 connectionstyle=cs))

# ============================ PANEL A ============================
gsL.text(-0.6, 9.7, "(a)", fontsize=15, fontweight="bold", va="top")

# --- network sits in the upper band (y ~ 4.5-9) ---
ins_y = [8.4, 7.3, 6.2, 5.1]
labels = ["cue", "distractor", "distractor", r"$\vdots$"]
incol = [CUE, DIST, DIST, DIST]
ix = 1.5
out = (6.4, 6.75)
synx = (ix + out[0]) / 2          # synapse-box column
bus_x = 8.3                        # reward bus column (right of everything)

for y, lab, c in zip(ins_y, labels, incol):
    if lab == r"$\vdots$":
        gsL.text(ix, y, r"$\vdots$", fontsize=22.4, ha="center", va="center", color=INK)
        continue
    gsL.add_patch(Circle((ix, y), 0.40, facecolor="white", edgecolor=c, lw=2.2, zorder=4))
    gsL.text(ix - 0.70, y, lab, fontsize=14, ha="right", va="center", color=c)
    # synapse box on the wire to the output
    gsL.add_patch(FancyBboxPatch((synx-0.34, y-0.21), 0.68, 0.42,
                  boxstyle="round,pad=0.02,rounding_size=0.06",
                  facecolor=DEV, edgecolor=INK, lw=1.2, zorder=5))
    gsL.plot([ix+0.40, synx-0.34], [y, y], color=c, lw=1.5, zorder=3)
    arrow(gsL, (synx+0.34, y), (out[0]-0.55, out[1]), color=c, lw=1.5, ms=10, z=3)
gsL.text(synx, ins_y[0]+0.62, "device synapses", fontsize=12.6,
         ha="center", color=INK, style="italic")

# LIF output neuron
gsL.add_patch(Circle(out, 0.55, facecolor=OUT, edgecolor=INK, lw=2.4, zorder=4))
gsL.text(out[0], out[1], "LIF", fontsize=14, ha="center", va="center", color=INK, fontweight="bold")
gsL.text(out[0], out[1]-0.85, "output", fontsize=13.3, ha="center", va="top", color=INK)

# --- reward delivered on a single clean vertical bus (no crossing lines) ---
# bus runs down the right side; a short horizontal stub taps each synapse box.
bus_top = ins_y[0] + 0.05
bus_bot = 3.7
gsL.plot([bus_x, bus_x], [bus_bot, bus_top], color=REW, lw=2.0, zorder=2)
for y in ins_y[:3]:
    arrow(gsL, (bus_x, y), (synx+0.36, y), color=REW, lw=1.2, ms=9,
          ls=(0, (3, 2)), z=2)
    gsL.add_patch(Circle((bus_x, y), 0.06, facecolor=REW, edgecolor=REW, zorder=3))
# reward source box feeding the bus from below (centred on the bus, fully inside axes)
rb_w, rb_h = 2.8, 1.0
rb_cx, rb_cy = bus_x, 3.15
gsL.add_patch(FancyBboxPatch((rb_cx-rb_w/2, rb_cy-rb_h/2), rb_w, rb_h,
              boxstyle="round,pad=0.03,rounding_size=0.1",
              facecolor="white", edgecolor=REW, lw=2.0, zorder=5))
gsL.text(rb_cx, rb_cy+0.18, r"reward $R(t)$", fontsize=14, ha="center", va="center", color=REW, zorder=6)
gsL.text(rb_cx, rb_cy-0.22, "global third factor", fontsize=11.2, ha="center", va="center", color=REW, style="italic", zorder=6)
gsL.plot([bus_x, bus_x], [rb_cy+rb_h/2, bus_bot], color=REW, lw=2.0, zorder=2)
gsL.text(bus_x+0.18, (bus_bot+bus_top)/2, "broadcast", fontsize=10.5, ha="left",
         va="center", color=REW, rotation=90, style="italic")

# --- timeline in its own band at the bottom (y ~ 0.4-2.0), well separated ---
ty = 1.1
gsL.annotate("", xy=(9.8, ty), xytext=(0.4, ty),
             arrowprops=dict(arrowstyle="-|>", color=INK, lw=1.3))
gsL.text(9.7, ty-0.45, "time", fontsize=11.9, ha="right", color=INK)
# cue epoch block (below the axis), label below it
gsL.add_patch(Rectangle((1.0, ty-0.30), 1.8, 0.30, facecolor=CUE, alpha=0.45, edgecolor="none"))
gsL.text(1.9, ty-0.55, "cue epoch", fontsize=11.9, ha="center", va="top", color=CUE)
# delay span (above the axis), label above it
gsL.annotate("", xy=(7.0, ty+0.30), xytext=(2.8, ty+0.30),
             arrowprops=dict(arrowstyle="<->", color=INK, lw=1.1))
gsL.text(4.9, ty+0.42, r"action$\to$reward delay $D$", fontsize=11.9, ha="center", va="bottom", color=INK)
# reward marker (on the axis), label above the tick, clear of the delay span
gsL.plot([7.0, 7.0], [ty-0.18, ty+0.18], color=REW, lw=2.4)
gsL.text(7.0, ty-0.30, "reward", fontsize=11.9, ha="center", va="top", color=REW)

# ============================ PANEL B ============================
gsR.text(-0.6, 9.7, "(b)", fontsize=15, fontweight="bold", va="top")
# 1. coincidence
gsR.add_patch(FancyBboxPatch((0.5, 8.0), 9.0, 1.05, boxstyle="round,pad=0.03,rounding_size=0.1",
              facecolor="white", edgecolor=INK, lw=1.6))
gsR.text(5.0, 8.72, "pre--post coincidence (signed, leak-dominant)", fontsize=10.8, ha="center", color=INK)
gsR.text(2.6, 8.25, r"causal: $+1$", fontsize=11.9, ha="center", color=CUE)
gsR.text(7.2, 8.25, r"acausal: $-\lambda$", fontsize=11.9, ha="center", color=REW)
arrow(gsR, (5.0, 8.0), (5.0, 7.35), lw=1.6)
# 2. device gate box (widened to match the coincidence box so the subtitle fits)
gsR.add_patch(FancyBboxPatch((0.5, 5.1), 9.0, 2.1, boxstyle="round,pad=0.03,rounding_size=0.1",
              facecolor=DEV, edgecolor=INK, lw=1.8))
gsR.text(5.0, 6.85, "device transient gate", fontsize=13.3, ha="center", color=INK, fontweight="bold")
gsR.text(5.0, 6.40, r"trap cascade ($k\!\approx\!3$, rise) $+$ $R_{\mathrm{leak}}$ (relax)", fontsize=10.8, ha="center", color=INK)
# mini trace inside
tx = np.linspace(0, 1, 120)
e = (1 - np.exp(-(tx/0.18)**2)) * np.exp(-tx/0.45); e = e/e.max()
gsR.plot(2.2 + tx*5.0, 5.35 + e*0.75, color=TRACE, lw=1.8)
gsR.text(7.6, 5.7, r"$e(t)$", fontsize=12.6, color=TRACE, ha="left")
arrow(gsR, (5.0, 5.1), (5.0, 4.45), lw=1.6)
# 3. eligibility trace label
gsR.text(5.0, 4.12, r"eligibility trace $e_{ij}(t)$,  retention $\tau_{\mathrm{leak}}\!=\!R_{\mathrm{leak}}C$",
         fontsize=12.3, ha="center", color=TRACE)
arrow(gsR, (5.0, 3.75), (5.0, 3.05), lw=1.6)
# 4. update box
gsR.add_patch(FancyBboxPatch((1.0, 1.6), 8.0, 1.35, boxstyle="round,pad=0.03,rounding_size=0.1",
              facecolor="white", edgecolor=REW, lw=2.0))
gsR.text(5.0, 2.50, "three-factor update at reward", fontsize=13.3, ha="center", color=REW, fontweight="bold")
gsR.text(5.0, 1.92, r"$\Delta w_{ij} = \eta\,(R-b)\,e_{ij}(t_R)$", fontsize=16.8, ha="center", color=INK)
finish(fig, "fig_architecture.png", provenance="immutable-authored", claim_status="authored-schematic-reproduction", bbox_inches="tight");

## Electron-scale eligibility mechanism

Authored schematic generator retained from the authored source.

In [ ]:
import os
import numpy as np
import matplotlib

import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, Circle, Rectangle, FancyBboxPatch

INK   = "#2b2b2b"
CB    = "#cdd6df"     # conduction band shade
TRAP  = "#3aa07a"     # trap level / trapped electrons / trace
ELEC  = "#3aa07a"
CAP   = "#c0392b"     # reward / capture-write
GREEN = "#c75c2e"     # coincidence (spike accent, distinct from teal trace)
NEU   = "#9aa6b2"
PAL   = [plt.cm.viridis(x) for x in (0.85, 0.5, 0.15)]   # tau_leak curves: sequential viridis

def panel_a(ax):
    ax.text(-0.06, 1.10, "(a)", transform=ax.transAxes, fontsize=13, fontweight="bold")
    ax.set_title("Capture, then emission at a trap", fontsize=9.6, fontweight="bold", pad=4)
    ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis("off")

    # conduction band
    ax.fill_between([0.5, 9.5], 8.2, 10, color=CB, zorder=1)
    ax.plot([0.5, 9.5], [8.2, 8.2], color=INK, lw=1.4, zorder=2)
    ax.text(9.4, 9.1, "conduction band", ha="right", fontsize=8.2, color=INK)

    # A short SEQUENTIAL cascade of trap sites at energetically-distributed depths
    # (faithful to Ch5: carriers hop through a chain v_n1 -> v_n2 -> ... -> v_nk;
    # the sites are spatially and energetically distributed, giving a SPREAD of
    # release times on discharge). Capture fills the chain; emission empties it.
    traps = [(2.2, 6.7), (4.4, 5.7), (6.6, 4.4)]   # distributed depths along the chain
    for j, (x, y) in enumerate(traps):
        ax.plot([x-0.5, x+0.5], [y, y], color=TRAP, lw=2.2, zorder=3)
        ax.add_patch(Circle((x, y+0.17), 0.15, fc=ELEC, ec=INK, lw=0.8, zorder=5))
        ax.text(x, y-0.5, rf"$v_{{n{j+1}}}$", ha="center", fontsize=7.4, color=TRAP)
        # sequential hop to the next site (the cascade rise)
        if j < len(traps)-1:
            xn, yn = traps[j+1]
            ax.add_patch(FancyArrowPatch((x+0.5, y), (xn-0.5, yn), arrowstyle="-|>",
                         mutation_scale=10, color=GREEN, lw=1.4,
                         connectionstyle="arc3,rad=-0.15", zorder=4))
    # capture from band into the chain head (during coincidence)
    ax.add_patch(FancyArrowPatch((2.2, 8.1), (2.2, 6.95), arrowstyle="-|>",
                 mutation_scale=12, color=GREEN, lw=1.7, zorder=4))
    # emission back to band, once the drive stops (dispersive: a spread of rates)
    for (x, y) in traps:
        ax.add_patch(FancyArrowPatch((x+0.30, y+0.30), (x+0.30, 8.1), arrowstyle="-|>",
                     mutation_scale=10, color=CAP, lw=1.2, ls=(0,(2,1.3)), zorder=4))
    # depth bracket: distribution of trap energies
    ax.annotate("", xy=(8.6, 8.2), xytext=(8.6, 4.4),
                arrowprops=dict(arrowstyle="<->", color=INK, lw=1.0))
    ax.text(8.8, 6.3, r"spread of$\ E_T$", fontsize=8.0, color=INK, rotation=90, va="center")
    # labels
    ax.text(2.0, 1.45, "capture (coincidence):", color=GREEN, fontsize=8.0, ha="center")
    ax.text(2.0, 0.95, "fills the sequential chain", color=GREEN, fontsize=7.2, ha="center")
    ax.text(6.8, 1.45, r"emission (drive off): rate $\propto e^{\gamma\sqrt{V}}$", color=CAP, fontsize=8.0, ha="center")
    ax.text(6.8, 0.95, "distributed $E_T$ $\\Rightarrow$ spread of release times", color=CAP, fontsize=7.2, ha="center")

def panel_b(ax):
    ax.text(-0.06, 1.10, "(b)", transform=ax.transAxes, fontsize=13, fontweight="bold")
    ax.set_title("Trapped population $=$ eligibility trace", fontsize=9.6, fontweight="bold", pad=4)
    dt = 0.01
    t = np.arange(0, 10, dt)
    tau, beta = 2.2, 0.6
    # build during a brief coincidence, then dispersive decay (clamp t-1>=0 so the
    # fractional power never sees a negative base)
    peak = 1 - np.exp(-(1.0/0.18)**2)
    td = np.maximum(t - 1.0, 0.0)
    e = np.where(t < 1.0, 1 - np.exp(-(t/0.18)**2),
                 peak*np.exp(-(td/tau)**beta))
    e = e / e.max()
    ax.fill_between(t, 0, e, color=TRAP, alpha=0.15)
    ax.plot(t, e, color=TRAP, lw=2.4)
    ax.axvspan(0, 1.0, color=GREEN, alpha=0.25)
    ax.text(0.5, 1.06, "coincidence", color=GREEN, fontsize=7.8, ha="center")
    # tau_leak marker at 1/e of peak
    ax.annotate("", xy=(1.0+tau, 0.05), xytext=(1.0, 0.05),
                arrowprops=dict(arrowstyle="<->", color=INK, lw=1.0))
    ax.text(1.0+tau/2, 0.12, r"$\tau_{\mathrm{leak}}=R_{\mathrm{leak}}C$", ha="center", fontsize=8.2, color=INK)
    # (floating "occupancy of trapped electrons = e(t)" annotation removed)
    ax.set_xlim(0, 10); ax.set_ylim(0, 1.12)
    ax.set_xlabel("time after coincidence (s)", fontsize=8.5)
    ax.set_ylabel("trapped population $e(t)$", fontsize=8.5)
    ax.tick_params(labelsize=7.5)

def panel_c(ax):
    ax.text(-0.06, 1.10, "(c)", transform=ax.transAxes, fontsize=13, fontweight="bold")
    ax.set_title("Reward captures the surviving trace", fontsize=9.6, fontweight="bold", pad=4)
    dt = 0.01
    t = np.arange(0, 10, dt)
    tau, beta = 2.2, 0.6
    peak = 1 - np.exp(-(1.0/0.18)**2)
    td = np.maximum(t - 1.0, 0.0)
    e = np.where(t < 1.0, 1 - np.exp(-(t/0.18)**2),
                 peak*np.exp(-(td/tau)**beta))
    e = e / e.max()
    ax.fill_between(t, 0, e, color=TRAP, alpha=0.15)
    ax.plot(t, e, color=TRAP, lw=2.2)
    # two reward times
    for tR, ok in [(3.0, True), (8.0, False)]:
        eR = e[int(tR/dt)]
        ax.axvline(tR, color=CAP, lw=1.6, ls=(0, (4, 2)), alpha=1.0 if ok else 0.55)
        ax.plot([tR], [eR], "o", color=CAP, ms=6, zorder=5)
        ax.text(tR, 1.04, "reward", color=CAP, fontsize=7.6, ha="center", alpha=1.0 if ok else 0.6)
        # committed weight change bar (proportional to e(t_R))
        ax.add_patch(Rectangle((tR+0.12, 0.0), 0.5, eR*0.8, fc=CAP, ec="none",
                               alpha=0.8 if ok else 0.3, zorder=4))
    # (floating "within window..." and "too late..." annotations removed; the
    #  reward markers and the Delta-w equation below convey the same point)
    ax.text(5.0, -0.30, r"$\Delta w_{ij}=\eta\,(R-b)\,e_{ij}(t_R)$  [Eq.~(3)]",
            transform=ax.transAxes, ha="center", fontsize=8.4, color=CAP).set_in_layout(False)
    ax.set_xlim(0, 10); ax.set_ylim(0, 1.12)
    ax.set_xlabel("action$\\rightarrow$reward delay (s)", fontsize=8.5)
    ax.set_ylabel("eligibility $e(t)$", fontsize=8.5)
    ax.tick_params(labelsize=7.5)

def panel_d(ax):
    ax.text(-0.06, 1.10, "(d)", transform=ax.transAxes, fontsize=13, fontweight="bold")
    ax.set_title(r"$\tau_{\mathrm{leak}}$ sets the credit window", fontsize=9.6, fontweight="bold", pad=4)
    dt = 0.01
    t = np.arange(0, 10, dt)
    peak = 1 - np.exp(-(1.0/0.18)**2)
    for tau, col, lab in [(0.8, PAL[2], r"$\tau_{\mathrm{leak}}$ short (shallow traps)"),
                          (2.2, PAL[1], r"$\tau_{\mathrm{leak}}$ mid"),
                          (5.0, PAL[0], r"$\tau_{\mathrm{leak}}$ long (deep traps)")]:
        td = np.maximum(t - 1.0, 0.0)
        e = np.where(t < 1.0, 1 - np.exp(-(t/0.18)**2),
                     peak*np.exp(-(td/tau)**0.6))
        e = e/e.max()
        ax.plot(t, e, color=col, lw=2.2, label=lab)
    ax.axvspan(0, 1.0, color=GREEN, alpha=0.18)
    ax.set_xlim(0, 10); ax.set_ylim(0, 1.08)
    ax.set_xlabel("time after coincidence (s)", fontsize=8.5)
    ax.set_ylabel("eligibility $e(t)$", fontsize=8.5)
    ax.tick_params(labelsize=7.5)
    ax.legend(fontsize=7.2, loc="upper right", framealpha=0.92)
    # (floating "fast-forgetting <-> integrating" annotation removed; the legend
    #  of short/mid/long tau_leak curves already conveys this)

fig, axes = plt.subplots(2, 2, figsize=(10.0, 7.2))
panel_a(axes[0, 0]); panel_b(axes[0, 1])
panel_c(axes[1, 0]); panel_d(axes[1, 1])
fig.subplots_adjust(left=0.07, right=0.97, top=0.92, bottom=0.10, wspace=0.26, hspace=0.42)

finish(fig, "fig_eligibility_electron.png", provenance="immutable-authored", claim_status="authored-schematic-reproduction");

## Biological-to-silicon translation map

Authored schematic generator retained from the authored source.

In [ ]:
import os
import numpy as np
import matplotlib

import matplotlib.pyplot as plt
from matplotlib.patches import (FancyBboxPatch, FancyArrowPatch, Circle, Rectangle,
                                Patch, Ellipse, Polygon, Wedge)

INK    = "#2b2b2b"
BIO    = "#b07cc6"
AXON   = "#e7b27a"     # axon terminal
SPINE  = "#b07cc6"     # dendritic spine
TAGB   = "#3aa07a"     # chemical tag (green)
VES    = "#8c5a3c"     # vesicles
DOPA   = "#c0392b"     # dopamine / reward
LTP    = "#b07cc6"     # LTP growth
TRAP   = "#2f4b8f"     # trap / electron
LEAK   = "#c75c2e"     # dispersive leak
REW    = "#c0392b"     # global scalar (R-b)
ELEC   = "#9aa6b2"     # electrode
OX     = "#eef1f4"     # oxide
FIL    = "#6b4f2a"     # filament
PRED   = "#b07cc6"     # predicted
NEU    = "#9aa6b2"

# row y-centres (top -> bottom) and their times -- generously spaced
ROWS = [(13.4, "0 s",  "action"),
        (9.9,  "3 s",  "wait"),
        (6.4,  "5 s",  "reward"),
        (2.6,  "10 s", "capture")]
XL = 3.3     # biology column centre
XR = 10.7    # silicon column centre
XC = 7.0     # timeline x
SY = 0.9     # synapse glyph scale
SC = 1.05    # cell glyph scale

# ----------------------------------------------------------------- synapse glyph
def synapse(ax, cx, cy, grow=0.0, lit=False, tag=None, faded=False):
    """A conventional chemical synapse: a presynaptic axon terminal (bouton) with
    vesicles on top, a synaptic cleft, and a postsynaptic mushroom dendritic spine
    (head on a neck into the dendrite shaft) below, receptors in the head. Signal
    flows top->bottom. Late-LTP enlarges the spine head and adds receptors; the tag
    is a mark in the spine head."""
    s = SY
    # ---- presynaptic axon + bouton (top) ----
    ax.add_patch(Rectangle((cx-0.10*s, cy+1.05*s), 0.20*s, 0.55*s, fc=AXON, ec=INK, lw=1.2, zorder=3))  # axon stalk
    bout_w, bout_h = 0.95*s, 0.62*s
    if lit:
        ax.add_patch(Ellipse((cx, cy+0.62*s), bout_w+0.14*s, bout_h+0.14*s, fc="none", ec="#e0a93b", lw=2.6, zorder=2))
    ax.add_patch(Ellipse((cx, cy+0.62*s), bout_w, bout_h, fc=AXON, ec=INK, lw=1.6, zorder=4))            # bouton
    # vesicles clustered toward the active zone (bottom of the bouton)
    rng = np.random.RandomState(2)
    for i in range(5):
        vx = cx-0.30*s + 0.60*s*rng.rand(); vy = cy+0.45*s + 0.34*s*rng.rand()
        ax.add_patch(Circle((vx, vy), 0.075*s, fc="white", ec=VES, lw=1.0, zorder=5))
    # ---- synaptic cleft ----
    ax.add_patch(Rectangle((cx-0.50*s, cy+0.20*s), 1.0*s, 0.06*s, fc="none", ec="none", zorder=3))
    if lit:  # neurotransmitter released into the cleft
        for k in range(4):
            ax.add_patch(Circle((cx-0.28*s+0.56*s*rng.rand(), cy+0.20*s+0.06*s*rng.rand()),
                         0.05*s, fc=TAGB, ec="none", alpha=0.85, zorder=6))
    # ---- postsynaptic mushroom spine (head + neck into dendrite) ----
    head_r = (0.42 + 0.26*grow)*s
    head_y = cy - 0.10*s - head_r
    ax.add_patch(Rectangle((cx-0.22*s, head_y-0.55*s), 0.44*s, 0.5*s, fc=SPINE, ec=INK, lw=1.2, zorder=2))  # dendrite shaft
    ax.add_patch(Rectangle((cx-0.09*s, head_y-0.10*s), 0.18*s, 0.32*s,
                 fc=(LTP if grow > 0 else SPINE), ec=INK, lw=1.1, zorder=3))                               # neck
    ax.add_patch(Circle((cx, head_y), head_r, fc=(LTP if grow > 0 else SPINE), ec=INK, lw=1.6,
                 alpha=0.95 if grow > 0 else 1.0, zorder=4))                                               # head
    # receptors on the head facing the cleft (top arc; more when grown)
    nrec = 3 + int(round(3*grow))
    for a in np.linspace(-0.7, 0.7, nrec):
        rx = cx + head_r*np.sin(a); ry = head_y + head_r*np.cos(a)
        ax.add_patch(Rectangle((rx-0.045*s, ry-0.05*s), 0.09*s, 0.12*s, fc=INK, ec="none", zorder=5))
    # the chemical tag inside the spine head
    if tag is not None:
        tc = "#a9d9c5" if faded else TAGB
        ax.add_patch(Circle((cx, head_y), 0.15*s, fc=tc, ec=INK, lw=0.9, zorder=6,
                     alpha=0.5 if faded else 1.0))

# ----------------------------------------------------------------- SiOx cell glyph
def cell(ax, cx, cy, fil=0.0, trapped=0):
    s = SC
    ax.add_patch(Rectangle((cx-0.95*s, cy+0.50*s), 1.9*s, 0.20*s, fc=ELEC, ec=INK, lw=0.9, zorder=4))   # top electrode
    ax.add_patch(Rectangle((cx-0.95*s, cy-0.70*s), 1.9*s, 0.20*s, fc=ELEC, ec=INK, lw=0.9, zorder=4))   # Mo bottom
    ax.add_patch(Rectangle((cx-0.95*s, cy-0.50*s), 1.9*s, 1.00*s, fc=OX, ec=INK, lw=0.9, zorder=3))     # SiOx
    ax.text(cx-0.80*s, cy+0.40*s, r"SiO$_x$", fontsize=7.0, color=INK, va="top", zorder=6)
    # filament (thickness = non-volatile weight)
    fw = (0.10 + 0.42*fil)*s
    if fil > 0:
        ax.add_patch(Rectangle((cx+0.55*s-fw/2, cy-0.50*s), fw, 1.00*s, fc=FIL, ec=INK, lw=0.7, zorder=5))
    # trap sites (filled = trapped electrons), left of the filament
    rng = np.random.RandomState(7)
    for i in range(6):
        x = cx-0.62*s + 0.55*s*rng.rand(); y = cy-0.34*s + 0.68*s*rng.rand()
        filled = i < trapped
        ax.add_patch(Circle((x, y), 0.075*s, fc=(TRAP if filled else "white"), ec=TRAP, lw=1.0, zorder=6))

# ----------------------------------------------------------------- main
fig, ax = plt.subplots(figsize=(11.0, 13.4))
ax.set_xlim(0, 14); ax.set_ylim(0, 15.8); ax.axis("off")

# column headers (in the top margin, clear of the first row band which tops at 14.85)
ax.text(XL, 15.5, "BIOLOGY", ha="center", fontsize=15, fontweight="bold", color=BIO)
ax.text(XL, 15.12, "synaptic tagging and capture", ha="center", fontsize=9.5, color=BIO, style="italic")
ax.text(XR, 15.5, "SILICON", ha="center", fontsize=15, fontweight="bold", color=TRAP)
ax.text(XR, 15.12, "subthreshold SiO$_x$ memristor", ha="center", fontsize=9.5, color=TRAP, style="italic")

# segmented timeline: a separate arrow into each node, with the event name
# sitting in the GAP just above the node (no overlap with the arrow or label)
node_r = 0.56
seg_tops = [14.85] + [ROWS[i][0]-node_r for i in range(len(ROWS)-1)]
for (y, t, lab), top in zip(ROWS, seg_tops):
    # faint full-width row band
    ax.add_patch(Rectangle((0.4, y-1.72), 13.2, 3.36, fc="#f4f5f7", ec="none", zorder=0))
    # event-name label sits in the gap above the node, beside the segment
    ax.text(XC, (top+y+node_r)/2, lab, ha="center", va="center", fontsize=11,
            color=INK, fontweight="bold", style="italic", zorder=7,
            bbox=dict(boxstyle="round,pad=0.18", fc="white", ec="none"))
    # arrow for this segment, stopping at the node top
    ax.add_patch(FancyArrowPatch((XC, top), (XC, y+node_r), arrowstyle="-|>",
                 mutation_scale=20, color=INK, lw=2.2, zorder=2))
    # the node circle with the timestamp
    ax.add_patch(Circle((XC, y), node_r, fc="white", ec=INK, lw=2.0, zorder=6))
    ax.text(XC, y, t, ha="center", va="center", fontsize=11.5, fontweight="bold", color=INK, zorder=7)
# tail arrow below the last node
ax.add_patch(FancyArrowPatch((XC, ROWS[-1][0]-node_r), (XC, 0.95), arrowstyle="-|>",
             mutation_scale=20, color=INK, lw=2.2, zorder=2))
ax.text(XC, 0.6, "time", ha="center", fontsize=10.5, color=INK, fontweight="bold")

# ---------------- BIOLOGY ----------------
y = ROWS[0][0]; synapse(ax, XL, y, grow=0.0, lit=True, tag=TAGB)
ax.text(XL, y-1.18, "terminal fires; sets a transient tag", ha="center", fontsize=10.0, color=TAGB)
y = ROWS[1][0]; synapse(ax, XL, y, grow=0.0, tag=TAGB, faded=True)
ax.text(XL, y-1.18, "the chemical tag fades", ha="center", fontsize=10.0, color=TAGB)
y = ROWS[2][0]; synapse(ax, XL, y, grow=0.0, tag=TAGB)
for dx, dy in [(-1.7, 1.0), (-1.1, 1.25), (1.3, 1.05), (1.9, 0.65), (0.5, 1.35), (-0.3, 1.1)]:
    ax.add_patch(Circle((XL+dx, y+dy), 0.12, fc=DOPA, ec="none", alpha=0.85, zorder=5))
ax.add_patch(FancyArrowPatch((XL+0.9, y+1.0), (XL+0.1, y+0.2), arrowstyle="-|>",
             mutation_scale=12, color=DOPA, lw=1.5, zorder=6))
ax.text(XL, y-1.18, "dopamine binds only the surviving tag", ha="center", fontsize=10.0, color=DOPA)
y = ROWS[3][0]; synapse(ax, XL, y, grow=1.0, tag=None)
ax.text(XL, y-1.58, "late-LTP: the spine enlarges", ha="center", fontsize=10.0, color=LTP)

# ---------------- SILICON ----------------
y = ROWS[0][0]; cell(ax, XR, y, fil=0.28, trapped=6)
ax.add_patch(FancyArrowPatch((XR-1.8, y), (XR-1.0, y), arrowstyle="-|>",
             mutation_scale=13, color=TRAP, lw=1.7, zorder=6))
ax.text(XR, y-1.30, "LIF fires; electrons into deep traps", ha="center", fontsize=10.0, color=TRAP)
y = ROWS[1][0]; cell(ax, XR, y, fil=0.28, trapped=2)
for dx in (-0.45, -0.20, 0.05):
    ax.add_patch(FancyArrowPatch((XR+dx, y+0.40), (XR+dx, y+0.82), arrowstyle="-|>",
                 mutation_scale=9, color=LEAK, lw=1.2, ls=(0,(2,1)), zorder=6))
# dispersive-leak inset, tucked into the right margin beside this cell (not floating)
ix, iy, iw, ih = XR+1.25, y-0.45, 1.05, 0.95
tt = np.linspace(0, 1, 60); kww = np.exp(-(tt/0.35)**0.54)
ax.plot(ix+tt*iw, iy+kww*ih, color=LEAK, lw=2.0, zorder=6)
ax.plot([ix, ix, ix+iw], [iy+ih, iy, iy], color=NEU, lw=0.8, zorder=5)   # tiny axes
ax.text(ix+iw*0.55, iy+ih*0.78, r"$\beta\!<\!1$", fontsize=8.6, color=LEAK, ha="left")
ax.text(ix+iw*0.5, iy-0.22, "leak", fontsize=7.2, color=LEAK, ha="center", style="italic")
ax.text(XR, y-1.30, "electrons detrap: dispersive leak", ha="center", fontsize=10.0, color=LEAK)
y = ROWS[2][0]; cell(ax, XR, y, fil=0.28, trapped=2)
# the (R-b) pulse is gated by the surviving tag: it must meet the trapped
# electrons (left of the cell), not act on the filament directly -- the
# filament change is the consequence, shown in the next (capture) row
ax.add_patch(FancyArrowPatch((XR+1.8, y+0.95), (XR-0.25, y+0.18), arrowstyle="-|>",
             mutation_scale=14, color=REW, lw=2.3, zorder=7))
ax.text(XR+1.95, y+1.05, r"$(R\!-\!b)$", ha="center", fontsize=11, color=REW, fontweight="bold")
ax.text(XR, y-1.30, "global scalar meets trapped electrons", ha="center", fontsize=10.0, color=REW)
y = ROWS[3][0]; cell(ax, XR, y, fil=1.0, trapped=1)
ax.text(XR, y-1.30, "non-volatile weight committed", ha="center", fontsize=10.0, color=FIL)
ax.text(XR, y-1.72, r"three-factor write: $\Delta w=\eta(R-b)\,e(t_R)$", ha="center", fontsize=8.8, color=FIL, style="italic")

fig.suptitle("From a synaptic tag to a silicon tag: a biological-to-device translation map",
             fontsize=13.5, fontweight="bold", y=0.975)
fig.subplots_adjust(left=0.01, right=0.99, top=0.93, bottom=0.015)

finish(fig, "fig_bio_silicon_map.png", provenance="immutable-authored", claim_status="authored-schematic-reproduction");

## Tier 1 — trace-level credit window

Fresh samples use the preserved device trace-level model. The plotting body follows the publication generator, including its log delay axis and learned threshold.

In [ ]:
from mrl_trace.distal_reward import run_trace_window
from mrl_trace.model_specs import PRIMARY_MODEL_ID, device_model_spec
MODEL_SPECIFICATION = device_model_spec(PRIMARY_MODEL_ID)

d = _archive("tier1_gate_results.npy") if USE_ARCHIVED_RESULTS else None
if d is None:
    d = run_trace_window(seeds=PROFILE["tier1_seeds"])
    source_mode = "live"
else:
    source_mode = "archived-result"

delays = d["delays"]
ratios = d["ratios"]
ratios_ci = d.get("ratios_ci", None)

# Sequential viridis by retention (long->short); no-trace control grey.
_VIR = [plt.cm.viridis(x) for x in (0.85, 0.5, 0.15)]
STYLE = {"gate_tl20": (_VIR[0], r"$\tau_{\mathrm{leak}}=20$ s"),
         "gate_tl5":  (_VIR[1], r"$\tau_{\mathrm{leak}}=5$ s"),
         "gate_tl1":  (_VIR[2], r"$\tau_{\mathrm{leak}}=1$ s"),
         "none":      ("#9aa6b2", "no-trace")}
CR = "#c0392b"

fig, ax = plt.subplots(figsize=(3.4, 2.8))
x = np.array(delays, float)
for name, (c, lab) in STYLE.items():
    y = np.array(ratios[name], float)
    ax.plot(x, y, marker="o", ms=3.5, lw=1.5, color=c, label=lab,
            zorder=4 if name != "none" else 3)
    if ratios_ci is not None:
        lo = np.array([ci[0] for ci in ratios_ci[name]])
        hi = np.array([ci[1] for ci in ratios_ci[name]])
        ax.fill_between(x, lo, hi, color=c, alpha=0.18, lw=0, zorder=2)

ax.axhline(2.0, ls=":", color=CR, lw=1.1, zorder=1)
ax.text(x[-1], 2.05, "learned", fontsize=6.5, ha="right", va="bottom", color=CR)
ax.set_xscale("log")
ax.set_xticks(delays)
ax.set_xticklabels([str(dd) for dd in delays], fontsize=7)
ax.set_xlabel(r"action$\to$reward delay $D$ (s)", fontsize=8.5)
ax.set_ylabel("cue / distractor weight ratio", fontsize=8.5)
ax.tick_params(labelsize=7.5)
ax.set_ylim(top=5.7)
ax.legend(fontsize=6.3, frameon=False, loc="upper center", ncol=2,
          columnspacing=1.2, handlelength=1.5, borderaxespad=0.3)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(True, which="major", color="0.85", lw=0.5, zorder=0)
ax.grid(True, which="minor", axis="x", color="0.92", lw=0.4, zorder=0)
ax.set_axisbelow(True)
fig.tight_layout()
finish(fig, "fig_tier1_window.png",
       provenance="live-exact" if RUN_PROFILE == "publication" and source_mode == "live" else "live-reduced",
       claim_status="full-sweep reproduction" if RUN_PROFILE == "publication" and source_mode == "live" else "reduced-validation",
       data=d, seeds=d.get("n_seeds"), source_mode=source_mode);

## Tier 2 — full-spiking credit saturation

Every point is computed from Poisson spikes, a LIF neuron, and one online device gate per synapse. The reduced profile changes only sample count, trial count, and integration step; publication restores 20 seeds, 400 trials, and 1 ms. Independent cells run in spawn-safe CPU processes.

In [ ]:
from mrl_trace.distal_reward import run_spiking_saturation

d = _archive("tier2_results.npy") if USE_ARCHIVED_RESULTS else None
if d is None:
    if USE_ARCHIVED_RESULTS:
        print("No Tier-2 archive is present; computing the sweep live.")
    d = run_spiking_saturation(
        seeds=PROFILE["tier2_seeds"], trials=PROFILE["tier2_trials"],
        dt=PROFILE["tier2_dt"], workers=RESOLVED_WORKERS,
    )
    source_mode = "live"
else:
    source_mode = "archived-result"

delays = np.array(d["delays"], float)
sat = d["saturation"]
sat_seeds = d.get("sat_seeds", None)

_VIR = [plt.cm.viridis(x) for x in (0.85, 0.5, 0.15)]
STYLE = [("gate_tl10", _VIR[0], r"$\tau_{\mathrm{leak}}=10$ s"),
         ("gate_tl2", _VIR[1], r"$\tau_{\mathrm{leak}}=2$ s"),
         ("gate_tl0.5", _VIR[2], r"$\tau_{\mathrm{leak}}=0.5$ s")]
CR = "#c0392b"


def ci_band(name):
    if sat_seeds is None or int(d.get("n_seeds", 0)) < 2:
        return None, None
    rng = np.random.default_rng(0)
    los, his = [], []
    for D in d["delays"]:
        values = sat_seeds[name]
        v = np.asarray(values[D] if D in values else values[str(D)], float)
        idx = rng.integers(0, v.size, size=(10000, v.size))
        means = v[idx].mean(1)
        lo, hi = np.percentile(means, [2.5, 97.5])
        los.append(lo); his.append(hi)
    return np.array(los), np.array(his)


fig, ax = plt.subplots(figsize=(3.4, 2.8))
for name, c, lab in STYLE:
    y = np.array(sat[name], float)
    ax.plot(delays, y, marker="o", ms=3.5, lw=1.5, color=c, label=lab, zorder=4)
    lo, hi = ci_band(name)
    if lo is not None:
        ax.fill_between(delays, lo, hi, color=c, alpha=0.18, lw=0, zorder=2)

ax.axhline(0.5, ls=":", color=CR, lw=1.1, zorder=1)
ax.set_xscale("log")
ax.set_xticks(d["delays"])
ax.set_xticklabels([str(dd) for dd in d["delays"]], fontsize=7)
ax.set_xlabel(r"action$\to$reward delay $D$ (s)", fontsize=8.5)
ax.set_ylabel("cue saturation toward bound", fontsize=8.5)
ax.set_ylim(0, 1.02)
ax.tick_params(labelsize=7.5)
ax.legend(fontsize=6.5, frameon=False, loc="upper left",
          bbox_to_anchor=(0.02, 0.98), handlelength=1.5)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(True, which="major", color="0.85", lw=0.5, zorder=0)
ax.grid(True, which="minor", axis="x", color="0.92", lw=0.4, zorder=0)
ax.set_axisbelow(True)
fig.tight_layout()
finish(fig, "fig_tier2_saturation.png",
       provenance="live-exact" if RUN_PROFILE == "publication" and source_mode == "live" else "live-reduced",
       claim_status="full-sweep reproduction" if RUN_PROFILE == "publication" and source_mode == "live" else "reduced-validation",
       data=d, seeds=d.get("n_seeds"), source_mode=source_mode);

## Tier 3 — closed-loop learning curve

Fresh device-trace and no-trace bandit runs supply the per-trial arrays required by the publication generator. Delay-summary aggregates are not substituted for this curve.

In [ ]:
from mrl_trace.bandit import train

d = _archive("tier3_results.npy") if USE_ARCHIVED_RESULTS else None
if d is None:
    dev = train(2, 2, B=PROFILE["rl_seeds"], tau_leak=10.0, D=5.0,
                trials=PROFILE["rl_trials"])
    nt = train(2, 2, B=PROFILE["rl_seeds"], tau_leak=10.0, D=5.0,
               trials=PROFILE["rl_trials"], no_trace=True)
    d = {"curve_device": dev, "curve_notrace": nt, "D0": 5.0,
         "n_seeds": PROFILE["rl_seeds"]}
    source_mode = "live"
else:
    dev, nt = d["curve_device"], d["curve_notrace"]
    source_mode = "archived-result"

DEVC = "#3aa07a"; NTC = "#9aa6b2"; CR = "#c0392b"
W = 50


def running(rw_2d, w=W):
    cs = np.cumsum(np.insert(rw_2d, 0, 0.0, axis=1), axis=1)
    rr = (cs[:, w:] - cs[:, :-w]) / w
    return rr.mean(0), np.percentile(rr, 2.5, axis=0), np.percentile(rr, 97.5, axis=0)


fig, ax = plt.subplots(figsize=(3.4, 2.7))
for arr, c, lab in [(dev, DEVC, "device trace"), (nt, NTC, "no-trace")]:
    m, lo, hi = running(arr)
    x = np.arange(W, W + len(m))
    ax.plot(x, m, color=c, lw=1.7, label=lab, zorder=4)
    ax.fill_between(x, lo, hi, color=c, alpha=0.2, lw=0, zorder=2)
ax.axhline(0.5, ls="--", color=CR, lw=1.0, zorder=1, label="chance")
ax.set_xlabel("trial", fontsize=9)
ax.set_ylabel("reward rate", fontsize=9)
ax.set_ylim(0.3, 1.03)
ax.tick_params(labelsize=8)
ax.legend(fontsize=7.5, frameon=False, loc="center right",
          bbox_to_anchor=(1.0, 0.62), handlelength=1.5)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(True, which="major", color="0.85", lw=0.5, zorder=0)
ax.set_axisbelow(True)
fig.tight_layout()
finish(fig, "fig_rl_curve.png",
       provenance="live-exact" if RUN_PROFILE == "publication" and source_mode == "live" else "live-reduced",
       claim_status="full-sweep reproduction" if RUN_PROFILE == "publication" and source_mode == "live" else "reduced-validation",
       data=d, seeds=d.get("n_seeds"), source_mode=source_mode);

## Crossbar mapping

Authored schematic generator retained from the authored source.

In [ ]:
import os
import numpy as np
import matplotlib

import matplotlib.pyplot as plt
from matplotlib.patches import (Circle, FancyBboxPatch, FancyArrowPatch,
                                Rectangle)

INK = "#2b2b2b"; STATE = "#e0a93b"; ACT = "#2f4b8f"; REW = "#c0392b"
TRACE = "#3aa07a"; DEV = "#eaf0fb"; OUT = "#cfd8e0"; GRID = "#9aa6b2"

fig = plt.figure(figsize=(9.8, 4.3))
A = fig.add_axes([0.015, 0.04, 0.575, 0.92]); A.axis("off")
B = fig.add_axes([0.625, 0.04, 0.365, 0.92]); B.axis("off")
A.set_xlim(0, 11); A.set_ylim(0, 9.6)
B.set_xlim(0, 10); B.set_ylim(0, 9)

def arrow(ax, p0, p1, color=INK, lw=1.7, ms=12, ls="-", z=5):
    ax.add_patch(FancyArrowPatch(p0, p1, arrowstyle="-|>", mutation_scale=ms,
                 color=color, lw=lw, ls=ls, zorder=z, shrinkA=0, shrinkB=0))

# ============================ PANEL A: crossbar ============================
A.text(-0.2, 8.8, "(a)", fontsize=15, fontweight="bold", va="top")

rows_y = [7.1, 6.0]                       # two state wordlines
cols_x = [4.2, 5.7]                       # two action bitlines
row_x0, row_x1 = 1.7, 6.6
col_y0, col_y1 = 4.6, 7.6

# state input wordlines (rows)
state_lab = [r"$s_1$", r"$s_2$"]
for y, lab in zip(rows_y, state_lab):
    A.plot([row_x0, row_x1], [y, y], color=STATE, lw=2.0, zorder=2)
    A.add_patch(Circle((row_x0 - 0.45, y), 0.34, facecolor="white",
                       edgecolor=STATE, lw=2.0, zorder=4))
    A.text(row_x0 - 0.45, y, lab, fontsize=9.5, ha="center", va="center", color=STATE)
A.text(row_x0 - 0.95, (rows_y[0] + rows_y[1]) / 2, "state\ninputs", fontsize=8.4,
       ha="right", va="center", color=STATE, style="italic")

# action bitlines (cols) + compound-cell synapses at crosspoints
for x in cols_x:
    A.plot([x, x], [col_y0, col_y1], color=ACT, lw=2.0, zorder=2)

def compound_cell(ax, cx, cy):
    """One synapse drawn as its true compound cell: a non-volatile WEIGHT device (left,
    dark) + a subthreshold TRACE device (right, light) + a small active-gating element
    (the access transistor / selector) marked beneath. Compact so the 2x2 array stays
    legible while honestly showing two devices per synapse rather than one glyph."""
    # weight device (left square, filled)
    ax.add_patch(FancyBboxPatch((cx - 0.30, cy - 0.15), 0.27, 0.30,
                 boxstyle="round,pad=0.01,rounding_size=0.04",
                 facecolor=DEV, edgecolor=INK, lw=1.0, zorder=5))
    # trace device (right square, light fill, trace colour edge)
    ax.add_patch(FancyBboxPatch((cx + 0.03, cy - 0.15), 0.27, 0.30,
                 boxstyle="round,pad=0.01,rounding_size=0.04",
                 facecolor="white", edgecolor=TRACE, lw=1.3, zorder=5))
    # active-gating element: a small filled circle (transistor/selector) under the pair,
    # tapping the third-factor line
    ax.add_patch(Circle((cx, cy - 0.32), 0.07, facecolor=REW, edgecolor=INK,
                        lw=0.8, zorder=6))

for y in rows_y:
    for x in cols_x:
        compound_cell(A, x, y)
# compact inline legend to the RIGHT of the array (clear of the left-margin labels),
# stacked above the I=G^T V annotation
leg_x = cols_x[1] + 1.15
leg_y = col_y1 + 0.55
A.add_patch(FancyBboxPatch((leg_x, leg_y - 0.10), 0.20, 0.22, boxstyle="round,pad=0.01",
            facecolor=DEV, edgecolor=INK, lw=0.8, zorder=5))
A.text(leg_x + 0.30, leg_y, "weight device", fontsize=6.8, ha="left", va="center", color=INK)
A.add_patch(FancyBboxPatch((leg_x, leg_y - 0.50), 0.20, 0.22, boxstyle="round,pad=0.01",
            facecolor="white", edgecolor=TRACE, lw=1.0, zorder=5))
A.text(leg_x + 0.30, leg_y - 0.40, "trace device", fontsize=6.8, ha="left", va="center", color=TRACE)
A.add_patch(Circle((leg_x + 0.10, leg_y - 0.80), 0.07, facecolor=REW, edgecolor=INK, lw=0.7, zorder=6))
A.text(leg_x + 0.30, leg_y - 0.80, "active gating", fontsize=6.8, ha="left", va="center", color=REW)
A.text(leg_x, leg_y + 0.40, "each synapse:", fontsize=7.0, ha="left", va="center",
       color=INK, style="italic")

# bitline current accumulation label
A.text(cols_x[1] + 1.15, (col_y0 + col_y1) / 2, r"$\mathbf{I}=\mathbf{G}^{\!\top}\mathbf{V}$",
       fontsize=10.5, ha="left", va="center", color=ACT)
A.text(cols_x[1] + 1.15, (col_y0 + col_y1) / 2 - 0.55, "(VMM along\nbitlines)",
       fontsize=7.6, ha="left", va="center", color=ACT, style="italic")

# action LIF neurons below the columns
act_y = 3.2
act_lab = [r"$a_1$", r"$a_2$"]
for x, lab in zip(cols_x, act_lab):
    arrow(A, (x, col_y0), (x, act_y + 0.5), color=ACT, lw=1.6, ms=10, z=3)
    A.add_patch(Circle((x, act_y), 0.42, facecolor=OUT, edgecolor=INK, lw=2.0, zorder=4))
    A.text(x, act_y, "LIF", fontsize=8.2, ha="center", va="center", color=INK, fontweight="bold", zorder=6)
    # label beside the neuron (not below) so it clears the WTA arrow on the bitline
    A.text(x + 0.55, act_y, lab, fontsize=9.0, ha="left", va="center", color=ACT, zorder=6)
A.text(cols_x[1] + 1.15, act_y - 0.5, "action\nneurons", fontsize=8.2, ha="left",
       va="center", color=ACT, style="italic")

# winner-take-all action selection
wta_y = 1.55
A.add_patch(FancyBboxPatch((cols_x[0] - 0.9, wta_y - 0.34), 3.3, 0.68,
            boxstyle="round,pad=0.03,rounding_size=0.1",
            facecolor="white", edgecolor=INK, lw=1.5, zorder=5))
A.text((cols_x[0] + cols_x[1]) / 2, wta_y, r"WTA: $a=\arg\max_j\!\sum_t s^{\mathrm{post}}_j$",
       fontsize=8.6, ha="center", va="center", color=INK, zorder=6)
for x in cols_x:
    arrow(A, (x, act_y - 0.45), (x, wta_y + 0.36), color=INK, lw=1.3, ms=8, z=3)

# environment / reward (contingent, delayed)
env_x = 9.55
A.add_patch(FancyBboxPatch((env_x - 1.15, wta_y - 0.42), 2.3, 1.5,
            boxstyle="round,pad=0.03,rounding_size=0.1",
            facecolor="white", edgecolor=REW, lw=1.8, zorder=5))
A.text(env_x, wta_y + 0.72, "environment", fontsize=8.6, ha="center", va="center", color=REW, zorder=6)
A.text(env_x, wta_y + 0.28, r"$R=\mathbf{1}(a{=}a^\star_s)$", fontsize=8.6, ha="center", va="center", color=INK, zorder=6)
A.text(env_x, wta_y - 0.16, r"delay $D$", fontsize=8.0, ha="center", va="center", color=REW, style="italic", zorder=6)
arrow(A, (cols_x[1] + 1.75, wta_y), (env_x - 1.2, wta_y), color=INK, lw=1.4, ms=10, z=3)

# global third-factor line: R-b broadcast back to every crosspoint
gf_y = 8.95
A.plot([env_x, env_x], [wta_y + 1.08, gf_y], color=REW, lw=1.6, ls=(0, (4, 2)), zorder=2)
A.plot([row_x0 - 0.1, env_x], [gf_y, gf_y], color=REW, lw=1.6, ls=(0, (4, 2)), zorder=2)
A.text((row_x0 + cols_x[1]) / 2 - 0.3, gf_y + 0.26,
       r"global third factor $(R-b)$ broadcast to all synapses",
       fontsize=8.2, ha="center", va="bottom", color=REW)
for x in cols_x:
    arrow(A, (x, gf_y), (x, rows_y[0] + 0.22), color=REW, lw=1.0, ms=7, ls=(0, (2, 2)), z=2)

# (generalization annotation removed -- the caption states the array is S x A)

# ============================ PANEL B: one crosspoint ============================
B.text(-0.2, 8.8, "(b)", fontsize=15, fontweight="bold", va="top")
B.text(5.0, 8.5, "single-layer synapse: one-term update", fontsize=9.0, ha="center", va="top",
       color=INK, fontweight="bold")

# weight device (non-volatile, switching regime) -- differential pair
B.add_patch(FancyBboxPatch((0.7, 5.7), 8.6, 1.9,
            boxstyle="round,pad=0.03,rounding_size=0.1",
            facecolor=DEV, edgecolor=INK, lw=1.5))
B.text(5.0, 7.25, "weight device (non-volatile, switching regime)", fontsize=8.2, ha="center", color=INK)
B.text(5.0, 6.45, r"$w_{ij}\;\propto\;G^{+}_{ij}-G^{-}_{ij}$", fontsize=11, ha="center", color=INK)
arrow(B, (5.0, 5.7), (5.0, 5.05), lw=1.5)

# trace device (subthreshold regime) -> local eligibility
B.add_patch(FancyBboxPatch((0.7, 3.0), 8.6, 2.0,
            boxstyle="round,pad=0.03,rounding_size=0.1",
            facecolor="white", edgecolor=TRACE, lw=1.6))
B.text(5.0, 4.6, "trace device (subthreshold regime)", fontsize=8.2, ha="center", color=TRACE)
tx = np.linspace(0, 1, 120)
e = (1 - np.exp(-(tx / 0.18) ** 2)) * np.exp(-tx / 0.45); e = e / e.max()
B.plot(1.5 + tx * 3.0, 3.35 + e * 0.85, color=TRACE, lw=1.8)
B.text(5.2, 3.75, r"local eligibility $e_{ij}(t)$", fontsize=8.6, ha="left", va="center", color=TRACE)
B.text(5.2, 3.30, r"retention $\tau_{\mathrm{leak}}$", fontsize=7.8, ha="left", va="center", color=TRACE, style="italic")
arrow(B, (5.0, 3.0), (5.0, 2.35), lw=1.5)

# access transistor performs the reward-gated multiply-write
B.add_patch(FancyBboxPatch((0.4, 0.7), 9.2, 1.6,
            boxstyle="round,pad=0.03,rounding_size=0.1",
            facecolor="white", edgecolor=REW, lw=1.9))
B.text(5.0, 1.92, "reward-gated multiply-write (active element)", fontsize=8.2, ha="center", color=REW)
B.text(5.0, 1.38, r"$\Delta w_{ij}=\eta\,(R-b)\;e_{ij}(t_R)$",
       fontsize=11.5, ha="center", color=INK)
B.text(2.95, 0.88, "global $(R-b)$", fontsize=7.4, ha="center", va="center", color=REW, style="italic")
B.text(6.55, 0.88, "local trace", fontsize=7.4, ha="center", va="center", color=TRACE, style="italic")
# ("no weight transport, no per-synapse gradient" removed -- stated in the text)
finish(fig, "fig_crossbar_rl.png", provenance="immutable-authored", claim_status="authored-schematic-reproduction", bbox_inches="tight");

## Notebook report

In [ ]:
assert len(FIGURE_REPORT) == 7
assert len({row["filename"] for row in FIGURE_REPORT}) == 7
assert all(row["saved_path"] is None for row in FIGURE_REPORT) if not SAVE_FIGURES else True
FIGURE_REPORT